In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# Load the Heart Disease dataset from UCI
url = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases"
    "/heart-disease/processed.cleveland.data"
)
column_names = [
    "age", "sex", "cp", "trestbps", "chol", "fbs",
    "restecg", "thalach", "exang", "oldpeak",
    "slope", "ca", "thal", "target"
]
df = pd.read_csv(url, names=column_names, na_values='?')

print('Shape:', df.shape)
print('\nFirst few rows:')
df.head()


Shape: (303, 14)

First few rows:


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


In [2]:
# Drop rows with missing values
df = df.dropna()
print('Shape after dropping NaN rows:', df.shape)

# Convert target to binary: 0 = no disease, 1 = disease
df['target'] = (df['target'] > 0).astype(int)
print('\nTarget distribution:')
print(df['target'].value_counts())

# Split features and target
X = df.drop('target', axis=1)
y = df['target']

# Train/test split with a fixed random seed
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features — important for logistic regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'\nTraining samples: {len(X_train)}')
print(f'Test samples:     {len(X_test)}')


Shape after dropping NaN rows: (297, 14)

Target distribution:
target
0    160
1    137
Name: count, dtype: int64

Training samples: 237
Test samples:     60


In [4]:
# ============================================================
# THE OLD WAY — no tracking, just print statements
# ============================================================

# Hyperparameters
C = 0.1          # regularization strength (lower = more regularization)
max_iter = 100   # maximum number of solver iterations
solver = 'lbfgs'

# Train the model
model = LogisticRegression(C=C, max_iter=max_iter, solver=solver, random_state=42)
model.fit(X_train_scaled, y_train)

# Evaluate
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print('=== Experiment Results ==='               )
print(f'C:        {C}')
print(f'max_iter: {max_iter}')
print(f'solver:   {solver}')
print(f'Accuracy: {accuracy:.4f}')
print(f'ROC AUC:  {roc_auc:.4f}')
print()
print(classification_report(y_test, y_pred))


=== Experiment Results ===
C:        0.1
max_iter: 100
solver:   lbfgs
Accuracy: 0.8500
ROC AUC:  0.9520

              precision    recall  f1-score   support

           0       0.83      0.91      0.87        32
           1       0.88      0.79      0.83        28

    accuracy                           0.85        60
   macro avg       0.85      0.85      0.85        60
weighted avg       0.85      0.85      0.85        60



#MLflow

In [5]:
import mlflow
import mlflow.sklearn
import os
os.chdir("..")


c:\Users\m6sbhatt\AppData\Local\anaconda3\envs\mlops-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# Name our experiment — MLflow will group all runs under this name
mlflow.set_experiment("heart-disease-classification")


2026/05/01 15:05:57 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/01 15:05:57 INFO mlflow.store.db.utils: Updating database tables
2026/05/01 15:05:58 INFO mlflow.tracking.fluent: Experiment with name 'heart-disease-classification' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///c:/ml-ops-tutorial/mlops-tutorial/mlruns/1', creation_time=1777673158551, experiment_id='1', last_update_time=1777673158551, lifecycle_stage='active', name='heart-disease-classification', tags={}, trace_location=None, workspace='default'>

In [7]:
# Hyperparameters for this run
C = 1.0
max_iter = 100
solver = 'lbfgs'

with mlflow.start_run(run_name='logistic-C1.0'):

    # Log the hyperparameters
    mlflow.log_param('C', C)
    mlflow.log_param('max_iter', max_iter)
    mlflow.log_param('solver', solver)

    # Train the model
    model = LogisticRegression(C=C, max_iter=max_iter, solver=solver, random_state=42)
    model.fit(X_train_scaled, y_train)

    # Evaluate
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)

    # Log the metrics
    mlflow.log_metric('accuracy', accuracy)
    mlflow.log_metric('roc_auc', roc_auc)

    # Log the model itself
    mlflow.sklearn.log_model(model, 'model')

    print(f'Run complete.')
    print(f'Accuracy: {accuracy:.4f}')
    print(f'ROC AUC:  {roc_auc:.4f}')


2026/05/01 15:07:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/01 15:07:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run complete.
Accuracy: 0.8333
ROC AUC:  0.9498


In [8]:
C = 0.1
max_iter = 200
solver = 'lbfgs'

with mlflow.start_run(run_name='logistic-C0.1'):

    mlflow.log_param('C', C)
    mlflow.log_param('max_iter', max_iter)
    mlflow.log_param('solver', solver)

    model = LogisticRegression(C=C, max_iter=max_iter, solver=solver, random_state=42)
    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)

    mlflow.log_metric('accuracy', accuracy)
    mlflow.log_metric('roc_auc', roc_auc)

    mlflow.sklearn.log_model(model, 'model')

    print(f'Run complete.')
    print(f'Accuracy: {accuracy:.4f}')
    print(f'ROC AUC:  {roc_auc:.4f}')



2026/05/01 15:08:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/01 15:08:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run complete.
Accuracy: 0.8500
ROC AUC:  0.9520
